# 5.11 同一段文字出現很多次會怎樣？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：先分整份文件，再切小窗口**

近似改寫的同一家族也放在同一側。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/split.svg")))

**先想一想：**「貓  看狗」和「貓 看狗」會得到同指紋嗎？

同一句話的空白差異可能掩蓋重複。先明示正規化規則，再hash比較；不要無條件刪掉內容有意義的空白。

**把直覺寫成數學：**hash是內容指紋；完全相同正規化字串才視為完全重複。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.data import fingerprint

docs = ["貓  看狗", "貓 看狗", "狗 看貓"]
hashes = [fingerprint(s) for s in docs]
print("原始", len(docs), "正規化不同內容", len(set(hashes)))
assert hashes[0] == hashes[1]

**如何讀結果：**程式用split/join合併空白；程式碼、詩排版或JSON可能需要別的規則。

**只改一件事：**只改一個字，觀察hash完全不同。
